In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from ingest import load_faq_data

In [4]:
documents = load_faq_data()

In [5]:
from sqlitesearch import TextSearchIndex


In [6]:
sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

In [7]:
docs_llm = [doc for doc in documents if doc["course"] == "llm-zoomcamp"]
print(f"LLM Zoomcamp: {len(docs_llm)} documents")

LLM Zoomcamp: 79 documents


In [7]:
for doc in docs_llm:
    sqlite_index.add(doc)
    print(f"Added document: {doc['question'][:50]}...")

sqlite_index.close()
print("Indexing complete.")

Added document: I just discovered the course. Can I still join?...
Added document: Course: I have registered for the LLM Zoomcamp. Wh...
Added document: What is the video/zoom link to the stream for the ...
Added document: Cloud alternatives with GPU...
Added document: Leaderboard: I am not on the leaderboard / how do ...
Added document: Certificate: Can I follow the course in a self-pac...
Added document: I missed the first homework - can I still get a ce...
Added document: Homework: Why does the content keep changing?...
Added document: When will the course be offered next?...
Added document: Are there any lectures/videos? Where are they?...
Added document: WSL2: ResponseError: model requires more system me...
Added document: Server Error (500) When logging in to course homew...
Added document: Why are we not using Langchain in the course?...
Added document: OpenAI: Error when running OpenAI responses.create...
Added document: OpenAI: Error: RateLimitError: Error code: 429 -...
Added

In [8]:
sqlite_index.count()

79

In [9]:
results = sqlite_index.search("Can I still join the course after it started?", num_results=5)
[doc["question"] for doc in results]

['I just discovered the course. Can I still join?',
 'I missed the first homework - can I still get a certificate?',
 'Do we submit 2 projects, what does attempt 1 and 2 mean?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'I am using Azure OpenAI and I am still getting an error of Error code: 400 - {\'error\': {\'message\': "Missing required parameter: \'tools[0].function\'.", \'type\': \'invalid_request_error\', \'param\': \'tools[0].function\', \'code\': \'missing_required_parameter\'}}?']

In [12]:
from rag_helper import RAGBase
from openai import OpenAI

openai_client = OpenAI()


assistant = RAGBase(
    index=sqlite_index,
    llm_client=openai_client,
)

In [22]:
answer = assistant.rag("Can I still join the course after it started?")
print(answer)

Yes, you can still join after it started. If you want a certificate, make sure to submit your project while submissions are still open.


In [32]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context. ensure user is asking relevent question about the course? and provide accurate answers based on the indexed FAQ data.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with humour if they are looking for something else."
"""

assistant = RAGBase(
    instruction=INSTRUCTIONS,
    index=sqlite_index,
    llm_client=openai_client,
)

In [33]:
answer = assistant.rag("Can I dance after it completed?")
print(answer)

If you mean **dance after it completed** — sure, celebrate away after finishing the course stuff. 🕺

If you mean **peer-review or certificate after the course is completed**:  
- **You can’t get a certificate in self-paced mode**.  
- **Peer review only happens while the course is running**, after the submission deadline, when projects are assigned to you.

So: **dance yes, certificate no** — unless you’re in the live cohort and do the required peer reviews.


In [ ]:
sqlite_index.close()